# Module 9 · 影片 1：影片 → 張量前處理（資料結構與解碼）

> **本節定位｜全新（2026 必備）**
> 影片是「**影格序列**」，比圖像多了一個**時間維度**。
> 現代影片模型（VideoMAE、ViViT、影片版 CLIP）的輸入是 5 維張量
> **`(N, T, C, H, W)`**。本節說明影片的資料結構與解碼。

## 學習目標
- 建立影片的資料結構觀：影格序列、時間維度 T、5 維張量 `(N, T, C, H, W)`。
- 用 PyAV / torchvision 解碼影片成影格（概念 + 程式）。
- 認識影片任務的標籤格式（動作分類 / 時序定位 / 字幕）。

## 0. 資料結構設計：影片張量與標籤

| 物件 | 形狀 | 說明 |
|:--|:--|:--|
| 原始檔 | — | mp4/avi（壓縮編碼，需解碼） |
| 解碼影格 | `(T, H, W, C)` | T 張 RGB 影格（numpy/PIL 慣例） |
| 單一 clip（PyTorch） | `(T, C, H, W)` | 已前處理的影格序列 |
| 模型輸入（批次） | `(N, T, C, H, W)` | **比圖像多一個時間維 T** |

**標籤格式（依任務）**：

| 任務 | 標籤 | 範例 |
|:--|:--|:--|
| 動作分類 | `(N,)` | 整段一個動作類別 |
| 時序動作定位 | 每片 `list[(start, end, cls)]` | 動作發生的時間區間 |
| 影片字幕 | `list[str]` | 描述文字（再經 tokenizer） |

In [1]:
import numpy as np

# 合成一段「影片」：16 張 64x64 RGB 影格（無需下載任何影片檔）
T, H, W, C = 16, 64, 64, 3
rs = np.random.RandomState(0)
# 讓亮度隨時間遞增，模擬「會動」的內容
frames = np.stack([(rs.rand(H, W, C) * 0.3 + i / T * 0.7) for i in range(T)])
frames = (frames * 255).astype(np.uint8)
print(f"解碼後影格: {frames.shape}  (T, H, W, C) ← numpy/解碼慣例")
print(f"T={T} 張影格，每張 {H}x{W} RGB")

解碼後影格: (16, 64, 64, 3)  (T, H, W, C) ← numpy/解碼慣例
T=16 張影格，每張 64x64 RGB


## 1. 真實影片解碼（PyAV / torchvision）

實務用 **PyAV (`av`)** 或 `torchvision.io.read_video` 解碼 mp4。需 `multimodal` extra。
（此處示範 API；無影片檔時會跳過。）

In [2]:
try:
    import torch
    import torchvision

    def decode_video(path, max_frames=16):
        # torchvision.io.read_video 回傳 (video, audio, info)，video 形狀 (T, H, W, C)
        video, _, info = torchvision.io.read_video(path, output_format="THWC")
        return video[:max_frames]

    print("torchvision.io.read_video → (T, H, W, C)；PyAV 可逐幀串流解碼超長影片。")
    print("（本節用合成影格示範後續流程，不實際讀檔。）")
    HAS = True
except Exception as e:
    HAS = False
    print("未安裝 torchvision，請 `uv sync --extra multimodal`。", e)

未安裝 torchvision，請 `uv sync --extra multimodal`。 No module named 'torch'


## 2. 影格 → PyTorch 張量 `(T, C, H, W)` → 批次 `(N, T, C, H, W)`

In [3]:
if HAS:
    import torch
    # (T, H, W, C) uint8 → (T, C, H, W) float[0,1]
    clip = torch.from_numpy(frames).permute(0, 3, 1, 2).float() / 255.0
    print(f"單一 clip: {tuple(clip.shape)}  (T, C, H, W)")
    batch = torch.stack([clip, clip])              # 兩段 clip
    print(f"批次張量: {tuple(batch.shape)}  (N, T, C, H, W) ← 影片模型的標準輸入")

## 小結

| 重點 | 內容 |
|:--|:--|
| 影片本質 | 影格序列，多一個時間維 T |
| 解碼輸出 | `(T, H, W, C)`（PyAV / torchvision.io） |
| 模型輸入 | `(N, T, C, H, W)`（比圖像多 T） |
| 標籤 | 動作分類 `(N,)`／時序定位／字幕 |

**下一節 `02_frame_sampling`**：影片很長且冗餘，不可能用全部影格——
需要**影格抽樣策略**，並用 VideoMAE 的 processor 標準化。